In [1]:
import pandas as pd
import numpy as np

movies = pd.read_csv('the-movies-dataset/movies_metadata.csv', low_memory=False)
more_movies = pd.read_csv('the-movies-dataset/TMDB _IMDB_Movies_Dataset.csv', low_memory=False)
keywords = pd.read_csv('the-movies-dataset/keywords.csv')
credits = pd.read_csv('the-movies-dataset/credits.csv')
movies['id'] = movies['id'].astype('int64')
movies = movies.merge(keywords, on='id')
movies = movies.merge(credits, on='id')


In [2]:
# Remove duplicates and rows with missing overviews
movies = movies.drop_duplicates()
more_movies = more_movies.drop_duplicates()

In [3]:
print(movies.overview.isnull().sum())
print(more_movies.overview.isnull().sum())

954
44945


In [4]:
# Drop missing overviews
movies = movies.dropna(subset=['overview'])
more_movies = more_movies.dropna(subset=['overview'])

In [5]:
# Convert the 'overview' column to a list of words, and the 'genres', 'cast', 'writers', and 'directors' columns to lists of strings
more_movies['overview'] = more_movies['overview'].apply(lambda x: '' if pd.isna(x) else x.split())
more_movies['genres'] = more_movies['genres'].apply(lambda x: [] if pd.isna(x) else x.split(', '))
more_movies['cast'] = more_movies['cast'].apply(lambda x: [] if pd.isna(x) else x.split(', '))
more_movies['writers'] = more_movies['writers'].apply(lambda x: [] if pd.isna(x) else x.split(', '))
more_movies['directors'] = more_movies['directors'].apply(lambda x: [] if pd.isna(x) else x.split(', '))

In [6]:
# Create a new column 'crew' that combines the 'directors', and 'writers' columns
more_movies['crew'] = more_movies['directors'] + more_movies['writers']
# Create a new column 'imdb_id' that is equal to the 'tconst' column
more_movies['imdb_id'] = more_movies['tconst']

In [7]:
# Drop unnecessary columns
movies = movies[['id', 'imdb_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
more_movies = more_movies[['id', 'imdb_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.info()
more_movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44502 entries, 0 to 46627
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        44502 non-null  int64 
 1   imdb_id   44487 non-null  object
 2   title     44502 non-null  object
 3   overview  44502 non-null  object
 4   genres    44502 non-null  object
 5   keywords  44502 non-null  object
 6   cast      44502 non-null  object
 7   crew      44502 non-null  object
dtypes: int64(1), object(7)
memory usage: 3.1+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 398578 entries, 0 to 443522
Data columns (total 8 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        398578 non-null  int64 
 1   imdb_id   398578 non-null  object
 2   title     398578 non-null  object
 3   overview  398578 non-null  object
 4   genres    398578 non-null  object
 5   keywords  164196 non-null  object
 6   cast      398578 non-null  object
 7   crew  

In [8]:
movies.iloc[0].genres

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [9]:
import ast
def convert(obj, getfield='name', conditionIndex=None, conditionField=None, top=-1): # convert stringified list of dictionaries to list of names
    L = []
    for i in ast.literal_eval(obj):
        if conditionIndex is None or (conditionIndex is not None and conditionField is not None and i[conditionIndex] == conditionField):
            if i[getfield] not in L:
                L.append(i[getfield])
    if top > 0:
        L = L[:top]
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert, top=3) # get top 3 cast members
movies['crew'] = movies['crew'].apply(lambda x: list(dict.fromkeys(convert(x, conditionIndex='job', conditionField='Director') + convert(x, conditionIndex='job', conditionField='Writer')))) # get directors and writers, remove duplicates

movies.head()

,id,imdb_id,title,overview,genres,keywords,cast,crew
0,862,tt0114709,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[Animation, Comedy, Family]","[jealousy, toy, boy, friendship, friends, riva...","[Tom Hanks, Tim Allen, Don Rickles]",[John Lasseter]
1,8844,tt0113497,Jumanji,When siblings Judy and Peter discover an encha...,"[Adventure, Fantasy, Family]","[board game, disappearance, based on children'...","[Robin Williams, Jonathan Hyde, Kirsten Dunst]",[Joe Johnston]
2,15602,tt0113228,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[Romance, Comedy]","[fishing, best friend, duringcreditsstinger, o...","[Walter Matthau, Jack Lemmon, Ann-Margret]","[Howard Deutch, Mark Steven Johnson]"
3,31357,tt0114885,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[Comedy, Drama, Romance]","[based on novel, interracial relationship, sin...","[Whitney Houston, Angela Bassett, Loretta Devine]",[Forest Whitaker]
4,11862,tt0113041,Father of the Bride Part II,Just when George Banks has recovered from his ...,[Comedy],"[baby, midlife crisis, confidence, aging, daug...","[Steve Martin, Diane Keaton, Martin Short]",[Charles Shyer]


In [10]:
movies.iloc[0].genres

['Animation', 'Comedy', 'Family']

In [11]:
more_movies.iloc[0].genres

['Action', 'Science Fiction', 'Adventure']

In [12]:
movies['overview'][0]

"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences."

In [13]:
# Append the title by removing spaces to the overview for better recommendations
movies['overview'] = movies['title'].apply(lambda x: str(x).replace(" ", "")) + " " + movies['overview']
# Remove null values from overview and convert to list of words
movies['overview'] = movies['overview'].apply(lambda x: [] if pd.isna(x) else x.split())

In [14]:
# Add more_movies to movies
movies = pd.concat([movies, more_movies], ignore_index=True)
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 443080 entries, 0 to 443079
Data columns (total 8 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        443080 non-null  int64 
 1   imdb_id   443065 non-null  object
 2   title     443080 non-null  object
 3   overview  443080 non-null  object
 4   genres    443080 non-null  object
 5   keywords  208698 non-null  object
 6   cast      443080 non-null  object
 7   crew      443080 non-null  object
dtypes: int64(1), object(7)
memory usage: 27.0+ MB


In [15]:
print("Total movies:", len(movies))
print("Duplicate IDs:", movies['id'].duplicated().sum())

Total movies: 443080
Duplicate IDs: 53632


In [16]:
# Remove duplicates based on 'id' and keep the first occurrence
movies = movies.drop_duplicates(subset='id', keep='first')
movies['overview'] = movies['overview'].apply(lambda x: x if isinstance(x, list) else [x] if isinstance(x, str) else [])

In [17]:
# Remove spaces from names in genres, keywords, cast, and crew
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x] if isinstance(x, list) else [])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])
movies.head()

,id,imdb_id,title,overview,genres,keywords,cast,crew
0,862,tt0114709,Toy Story,"[ToyStory, Led, by, Woody,, Andy's, toys, live...","[Animation, Comedy, Family]","[jealousy, toy, boy, friendship, friends, riva...","[TomHanks, TimAllen, DonRickles]",[JohnLasseter]
1,8844,tt0113497,Jumanji,"[Jumanji, When, siblings, Judy, and, Peter, di...","[Adventure, Fantasy, Family]","[boardgame, disappearance, basedonchildren'sbo...","[RobinWilliams, JonathanHyde, KirstenDunst]",[JoeJohnston]
2,15602,tt0113228,Grumpier Old Men,"[GrumpierOldMen, A, family, wedding, reignites...","[Romance, Comedy]","[fishing, bestfriend, duringcreditsstinger, ol...","[WalterMatthau, JackLemmon, Ann-Margret]","[HowardDeutch, MarkStevenJohnson]"
3,31357,tt0114885,Waiting to Exhale,"[WaitingtoExhale, Cheated, on,, mistreated, an...","[Comedy, Drama, Romance]","[basedonnovel, interracialrelationship, single...","[WhitneyHouston, AngelaBassett, LorettaDevine]",[ForestWhitaker]
4,11862,tt0113041,Father of the Bride Part II,"[FatheroftheBridePartII, Just, when, George, B...",[Comedy],"[baby, midlifecrisis, confidence, aging, daugh...","[SteveMartin, DianeKeaton, MartinShort]",[CharlesShyer]


In [18]:
# Create a new column 'tags' that combines title,overview, genres, keywords, cast, and crew into a single string for each movie
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies.head()

,id,imdb_id,title,overview,genres,keywords,cast,crew,tags
0,862,tt0114709,Toy Story,"[ToyStory, Led, by, Woody,, Andy's, toys, live...","[Animation, Comedy, Family]","[jealousy, toy, boy, friendship, friends, riva...","[TomHanks, TimAllen, DonRickles]",[JohnLasseter],"[ToyStory, Led, by, Woody,, Andy's, toys, live..."
1,8844,tt0113497,Jumanji,"[Jumanji, When, siblings, Judy, and, Peter, di...","[Adventure, Fantasy, Family]","[boardgame, disappearance, basedonchildren'sbo...","[RobinWilliams, JonathanHyde, KirstenDunst]",[JoeJohnston],"[Jumanji, When, siblings, Judy, and, Peter, di..."
2,15602,tt0113228,Grumpier Old Men,"[GrumpierOldMen, A, family, wedding, reignites...","[Romance, Comedy]","[fishing, bestfriend, duringcreditsstinger, ol...","[WalterMatthau, JackLemmon, Ann-Margret]","[HowardDeutch, MarkStevenJohnson]","[GrumpierOldMen, A, family, wedding, reignites..."
3,31357,tt0114885,Waiting to Exhale,"[WaitingtoExhale, Cheated, on,, mistreated, an...","[Comedy, Drama, Romance]","[basedonnovel, interracialrelationship, single...","[WhitneyHouston, AngelaBassett, LorettaDevine]",[ForestWhitaker],"[WaitingtoExhale, Cheated, on,, mistreated, an..."
4,11862,tt0113041,Father of the Bride Part II,"[FatheroftheBridePartII, Just, when, George, B...",[Comedy],"[baby, midlifecrisis, confidence, aging, daugh...","[SteveMartin, DianeKeaton, MartinShort]",[CharlesShyer],"[FatheroftheBridePartII, Just, when, George, B..."


In [19]:
new_df = movies[['id', 'imdb_id', 'title', 'tags']].copy() # create a new dataframe with only the necessary columns
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x)) # convert list of words to a single string
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower()) # convert to lowercase
new_df['tags'][0]

"toystory led by woody, andy's toys live happily in his room until andy's birthday brings buzz lightyear onto the scene. afraid of losing his place in andy's heart, woody plots against buzz. but when circumstances separate buzz and woody from their owner, the duo eventually learns to put aside their differences. animation comedy family jealousy toy boy friendship friends rivalry boynextdoor newtoy toycomestolife tomhanks timallen donrickles johnlasseter"

In [20]:
# Stemming the words in the 'tags' column to reduce them to their root form
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text):
    return " ".join(ps.stem(word) for word in text.split())

new_df['tags'] = new_df['tags'].apply(stem)

In [21]:
# Create a bag of words model using CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=20000,
    stop_words='english',
    ngram_range=(1, 2)
)

vectors = tfidf.fit_transform(new_df['tags'])

In [22]:
# Compute cosine similarity between all movies based on their tags
from sklearn.metrics.pairwise import cosine_similarity
def recommend(movie, top_n=5):
    matches = new_df[new_df['title'].str.lower() == movie.strip().lower()]

    if matches.empty:
        print(f"Movie '{movie}' not found!")
        return

    movie_index = matches.index[0]
    distances = cosine_similarity(vectors[movie_index], vectors).flatten()
    movie_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:top_n+1]
    for i in movie_list:
        print(f"{new_df.iloc[i[0]].title} - https://www.imdb.com/title/{new_df.iloc[i[0]].imdb_id}/ - https://www.themoviedb.org/movie/{new_df.iloc[i[0]].id}/")


In [23]:
# Recommend movies similar to 'batman'
recommend('batman', 10)

Batman Beyond: Return of the Joker - https://www.imdb.com/title/tt0233298/ - https://www.themoviedb.org/movie/16234/
Batman: Under the Red Hood - https://www.imdb.com/title/tt1569923/ - https://www.themoviedb.org/movie/40662/
Batman: Arkham Asylum (The Movie) - https://www.imdb.com/title/tt1282022/ - https://www.themoviedb.org/movie/1760650/
Batman Returns - https://www.imdb.com/title/tt0103776/ - https://www.themoviedb.org/movie/364/
Batman Unlimited: Monster Mayhem - https://www.imdb.com/title/tt4729754/ - https://www.themoviedb.org/movie/342917/
The Dark Knight - https://www.imdb.com/title/tt0468569/ - https://www.themoviedb.org/movie/155/
The Lego Batman Movie - https://www.imdb.com/title/tt4116284/ - https://www.themoviedb.org/movie/324849/
Batman: Gotham by Gaslight - https://www.imdb.com/title/tt7167630/ - https://www.themoviedb.org/movie/471474/
The Dark Knight Rises - https://www.imdb.com/title/tt1345836/ - https://www.themoviedb.org/movie/49026/
Batman Unmasked: The Psycholog

In [24]:
import pickle

pickle.dump(new_df.to_dict(), open('movies.pkl', 'wb'))
pickle.dump(vectors, open('vectors.pkl', 'wb'))